In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen1.5-MoE-A2.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,   
    device_map="auto"
)
model.eval()

/home/user2/miniforge3/envs/giangntt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 8/8 [00:05<00:00,  1.42it/s]


Qwen2MoeForCausalLM(
  (model): Qwen2MoeModel(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-23): 24 x Qwen2MoeDecoderLayer(
        (self_attn): Qwen2MoeSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2MoeRotaryEmbedding()
        )
        (mlp): Qwen2MoeSparseMoeBlock(
          (gate): Linear(in_features=2048, out_features=60, bias=False)
          (experts): ModuleList(
            (0-59): 60 x Qwen2MoeMLP(
              (gate_proj): Linear(in_features=2048, out_features=1408, bias=False)
              (up_proj): Linear(in_features=2048, out_features=1408, bias=False)
              (down_proj): Linear(in_features=1408, out_features=2048, bias=False)
        

In [2]:
from data_utils import create_packed_dataloader
loader = create_packed_dataloader(tokenizer, "brando/small-c4-dataset", split="train", sample_size=512, max_length=512)

In [3]:
# Register hooks to capture expert activations

def save_expert_activation(module, input, output):
    layer_id = getattr(module, 'layer_id', None)
    expert_id = getattr(module, 'expert_id', None)
    key = (layer_id, expert_id)
    expert_activations.setdefault(key, []).append(output.detach().cpu())

def register_hooks(model):
    for layer_idx, layer in enumerate(model.model.layers):
        moe_block = layer.mlp
        moe_block.layer_id = layer_idx

        if hasattr(moe_block, 'experts'):
            for expert_idx, expert in enumerate(moe_block.experts):
                down_proj_layer = expert.down_proj
                down_proj_layer.layer_id = layer_idx
                down_proj_layer.expert_id = expert_idx
                handle = down_proj_layer.register_forward_hook(save_expert_activation)
                hook_handles.append(handle)

def clear_hooks():
    for handle in hook_handles:
        handle.remove()
    hook_handles.clear()
                
expert_activations = {}
hook_handles = []
register_hooks(model)

In [4]:
from tqdm import tqdm

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(loader, desc="Inferring expert activations")):
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)

        # Forward pass
        _ = model(input_ids=input_ids, attention_mask=attention_mask)

clear_hooks()


Inferring expert activations:   0%|          | 0/36 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

In [ ]:
import torch

def compute_asymmetric_overlap_matrix(
    expert_activations,
    layer_id,
    rank=64,
    sample_tokens=2000,
    device='cuda'
):
    """
    Compute asymmetric overlap matrix for a given layer.

    This measures how much each expert's activations (source) are captured
    by the low-rank subspace of other experts (target). High values indicate
    redundancy: source expert i is largely represented by target expert j.

    Args:
        expert_activations: dict[(layer_id, expert_id)] -> list of tensors
            Each tensor: [num_tokens_batch, hidden_dim] for activations of that expert
        layer_id: int, the layer to analyze
        rank: int, rank of the subspace for projection
        sample_tokens: int, maximum tokens to sample per expert to speed up computation
        device: 'cuda' or 'cpu'

    Returns:
        asymmetric_overlap: torch.Tensor [num_experts, num_experts]
            asymmetric_overlap[i, j] = fraction of activations of expert i
            captured by the low-rank subspace of expert j
    """
    # Collect all expert IDs in this layer
    expert_ids = sorted([e for (l, e) in expert_activations.keys() if l == layer_id])
    num_experts = len(expert_ids)

    # Dictionaries to store low-rank bases and full activations
    expert_bases = {}
    expert_full_X = {}

    # Step 1: Construct low-rank bases and store full activations for each expert
    for e in expert_ids:
        # Concatenate all batches for this expert
        X = torch.cat(expert_activations[(layer_id, e)], dim=0)  # [num_tokens_total, hidden_dim]

        # Optional: randomly sample tokens for efficiency
        if sample_tokens and X.size(0) > sample_tokens:
            idx = torch.randperm(X.size(0))[:sample_tokens]
            X = X[idx]

        # Move to device
        X = X.to(device)

        # Center activations
        X_centered = X - X.mean(dim=0, keepdim=True)
        X_centered = X_centered.float()  # ensure float32 for SVD

        # Step 1a: Low-rank basis using SVD
        # U, S, Vh = torch.linalg.svd(X_centered)
        # Take top 'rank' singular vectors as basis
        U, S, Vh = torch.linalg.svd(X_centered, full_matrices=False)
        basis = Vh[:rank].T  # [hidden_dim, rank]

        expert_bases[e] = basis         # store low-rank basis
        expert_full_X[e] = X_centered   # store full centered activations

    # Step 2: Initialize asymmetric overlap matrix
    # Rows: source experts (i), Columns: target experts (j)
    asymmetric_overlap = torch.zeros((num_experts, num_experts), device=device)

    # Step 3: Compute overlap for each pair of experts
    for i in range(num_experts):
        Xi = expert_full_X[expert_ids[i]]  # full activations of source expert i
        for j in range(num_experts):
            Bj = expert_bases[expert_ids[j]]  # low-rank basis of target expert j

            # Project source activations onto target basis
            X_proj = Xi @ Bj @ Bj.T  # [num_tokens, hidden_dim]

            # Overlap = fraction of Xi captured by Bj
            overlap = torch.norm(X_proj, p='fro')**2 / torch.norm(Xi, p='fro')**2
            asymmetric_overlap[i, j] = overlap

    return asymmetric_overlap


In [ ]:
from visualization_utils import plot_matrix
layers_to_plot = [1, 6]  # Adjust based on model depth
for layer in layers_to_plot:
    asym_overlap = compute_asymmetric_overlap_matrix(
        expert_activations,
        layer_id=layer,
        rank=128,
        sample_tokens=2000,
        device='cuda'
    )
    plot_matrix(asym_overlap, title=f"Asymmetric Overlap - Layer {layer}", 
                xlabel="Expert ID", ylabel="Expert ID", cmap="Reds", annot=False)

In [11]:
import torch
from collections import defaultdict

def get_super_experts(expert_activations, num_layers, num_experts_per_layer, top_k_ratio=0.2):
    """
    Compute expert scores and select super experts per layer.

    Args:
        expert_activations (dict): {(layer_id, expert_id): [tensor(batch1), tensor(batch2), ...]}
                                   Each tensor shape: [num_tokens_batch, hidden_dim]
        num_layers (int): Total number of MoE layers
        num_experts_per_layer (int): Number of experts per layer
        top_k_ratio (float): Fraction of experts to keep as super experts per layer

    Returns:
        super_experts_per_layer (dict): {layer_id: [(expert_id, score), ...]}
    """
    # Compute per-expert scores
    expert_scores = {}
    for (layer_id, expert_id), batches in expert_activations.items():
        # Compute the max activation across batches in a memory-efficient way
        max_val = max(batch.abs().max().item() for batch in batches)
        expert_scores[(layer_id, expert_id)] = max_val

    # Sort experts per layer
    super_experts_per_layer = defaultdict(list)
    for layer in range(num_layers):
        # Filter experts for this layer
        layer_experts = [(expert_id, score) for (l, expert_id), score in expert_scores.items() if l == layer]
        # Sort by score descending
        layer_experts.sort(key=lambda x: x[1], reverse=True)
        # Keep top-k
        keep_num = int(num_experts_per_layer * top_k_ratio)
        super_experts_per_layer[layer] = layer_experts[:keep_num]

    return super_experts_per_layer


In [14]:
num_layers = 24
num_experts_per_layer = 60
top_k_ratio = 1  # keep all experts per layer

# Select super experts based on max activation
super_experts = get_super_experts(expert_activations, num_layers, num_experts_per_layer, top_k_ratio)

In [15]:
import json

# Save super experts dictionaries to JSON in two versions: with scores and only expert IDs.

# Save super experts with scores
with open("prune_experts/super_experts.json", "w") as f:
    json.dump(super_experts, f, indent=4)

# Save IDs only
super_experts_ids = {layer: [expert_id for expert_id, _ in experts] for layer, experts in super_experts.items()}

with open("prune_experts/super_experts_ids.json", "w") as f:
    json.dump(super_experts_ids, f, indent=4)


In [ ]:
from common_utils import get_topk_experts_from_json
experts_to_prune = get_topk_experts_from_json(
    path="prune_experts/super_experts_ids.json",
    top_k=10,  # select top 10 experts
    mode="least",
    criterion="magnitude")

Loaded 'prune_experts/super_experts_ids.json' and produced top-10 least used/important experts per layer (criterion=magnitude)
Layers: 24, total experts selected: 240


: 